# Colab launcher — TSP LLM generation loop

This notebook launches the TSP LLM-generation loop on the original feedback instances used during the thesis experiments.

It does **not** download TSPLIB files and does **not** generate POPMUSIC/LKH candidate or edge-prior cache files. The required `.tsp`, `.cand`, and `.npz` files are expected to already exist in Drive/local storage. If a large cache file is missing, build it separately on the school server with the dedicated server-side cache builder.


## 1. Run control panel

Most changes should happen here. This mirrors the clustering launcher style, but with TSP-specific controls.


In [19]:
# ============================================================
# RUN CONTROL PANEL
# ============================================================

# -------------------------
# GitHub backend repo
# -------------------------
ORG = "TM-HESSO-202526"
REPO = "llm-tsp-heuristics"
BRANCH = "main"

# -------------------------
# Main run choice
# -------------------------
# This launcher is for the LLM feedback loop, not for building large-instance caches.
# Required TSPLIB / POPMUSIC / edge-prior files must already exist in Drive/local storage.
RUN_NAME = "tsp_strict_constructive_llm_loop"
SMOKE_TEST = False                       # True = force exactly 1 LLM attempt
MAX_LLM_CALLS = 6                        # Small default for a quick strict-constructive check

# Cache building/downloading is intentionally disabled in the LLM loop.
CACHE_ONLY_BUILD_TSP_CACHES = False
DOWNLOAD_MISSING_TSPLIB = False

# -------------------------
# LLM provider and sampling
# -------------------------
LLM_PROVIDER = "groq"
LLM_MODEL = "llama-3.3-70b-versatile"
TEMPERATURE = 0.8
TOP_P = 1.0

# -------------------------
# Groq key/rate-limit behavior
# -------------------------
GROQ_MAX_KEYS = 10
LLM_CALLS_PER_MINUTE_PER_KEY = 2
LLM_REQUEST_TIMEOUT_S = 60
MAX_429_RETRIES = 100
MAX_REQUEST_ERROR_RETRIES = 5

# -------------------------
# Evolution/search behavior, matched to the clustering launcher
# -------------------------
SELECTION_STRATEGY = "1+1"       # "1+1" = elitist best-so-far parent; "1,1" = latest sequential parent
HISTORY_LIMIT = 20               # Number of previous attempts summarized inside the prompt history

# -------------------------
# Invalid-parent redesign behavior, matched to the clustering launcher
# -------------------------
INVALID_PARENT_REDESIGN = True   # If no full-valid parent exists, use a special redesign prompt for invalid/partial parents
REDESIGN_ON_ANY_INVALID_BEFORE_FULL_VALID = True  # Trigger redesign on invalid/partial parents before first full-valid heuristic
REDESIGN_ON_TIMEOUT_PARENT = True # Trigger redesign when the selected parent has timeout/runtime failures
HIDE_INVALID_PARENT_CODE = False # False = expose invalid/partial parent code for diagnosis; True = hide it from the LLM

# -------------------------
# Historical family avoidance
# When True, injects the fixed historical avoidance block into the LLM prompt.
# -------------------------
HISTORICAL_FAMILY_AVOIDANCE = False

# -------------------------
# Strict constructive-only mode
# When True, the LLM is explicitly forbidden from adding 2-opt, 3-opt,
# local search, segment reversal, relocate/swap, or any post-construction
# optimization/refinement/cleanup phase.
# -------------------------
STRICT_CONSTRUCTIVE_ONLY = True


# -------------------------
# Family-focus / island exploitation mode
# -------------------------
# When True, the run is split into one local 1+1/1,1 block per family.
# Each family block has its own local parent and local history.
# At the end, the backend compares the best candidate from each family.
#
# Total LLM calls in this mode = FAMILY_FOCUS_CALLS_PER_FAMILY × number of enabled families.
# You can disable/comment out individual families below to keep the run smaller.
FAMILY_FOCUS_MODE = False
FAMILY_FOCUS_CALLS_PER_FAMILY = 20

# Default focused rerun: only the three most useful families are enabled.
# Set "enabled": True on other families below if you want the full family sweep again.
FAMILY_FOCUS_PLAN = [
    {
        "id": "mst_skeleton",
        "enabled": False,
        "name": "MST / tree skeleton construction",
        "objective": "Build a sparse tree-like or forest-like skeleton, then traverse, patch, or shortcut it into one Hamiltonian tour.",
        "description": "This family was attempted in historical-family-avoidance runs but often timed out or collapsed into closest-unvisited construction. The goal is to make the tree/skeleton idea real and scalable.",
        "strict_constraints": [
            "The main object must be a spanning-tree-like or forest-like skeleton.",
            "Do not implement this as closest-unvisited nearest-neighbor from the current city.",
            "Avoid dense all-pairs MST construction on 1000+ nodes.",
            "Use bounded local edge queries, coordinate shortlists, or regional tree fragments instead of a full dense graph.",
            "Explicitly explain in code comments how the skeleton is converted into a valid Hamiltonian tour.",
        ],
    },
    {
        "id": "voronoi_regions",
        "enabled": True,
        "name": "Voronoi / region decomposition",
        "objective": "Partition the cities into geometric regions, construct local paths inside each region, and connect region endpoints into one Hamiltonian tour.",
        "description": "Previous runs often computed Voronoi-like regions but then ignored them. Here the region decomposition must actually determine the tour structure.",
        "strict_constraints": [
            "The Voronoi/region partition must actually determine the tour structure.",
            "Do not compute regions and then ignore them.",
            "The algorithm must explicitly solve local ordering inside regions and bridge selection between region endpoints.",
            "Do not fall back to a global closest-unvisited walk after building the regions.",
            "Region-to-region bridges should be selected deliberately, not by simply appending arbitrary region tours.",
        ],
    },
    {
        "id": "convex_hull_outside_in",
        "enabled": True,
        "name": "Convex hull / outside-in insertion",
        "objective": "Start from an outer boundary cycle or approximate hull, then insert interior cities in a structured outside-in way.",
        "description": "Previous hull attempts were either very weak or computed a hull and then switched to nearest-neighbor. This block focuses on making outside-in construction the true mechanism.",
        "strict_constraints": [
            "The main mechanism must be outside-in construction.",
            "Start from an outer boundary cycle, approximate hull, or layered hull structure.",
            "Insert interior cities using insertion delta, edge impact, or geometric distance to tour edges.",
            "Do not switch to nearest-neighbor after computing the hull.",
            "Handle the closing edge of the tour correctly when inserting cities.",
        ],
    },
    {
        "id": "grid_sector_decomposition",
        "enabled": True,
        "name": "Grid / sector decomposition",
        "objective": "Divide the plane into grid cells or angular sectors, order the regions, build local paths, and connect endpoints.",
        "description": "This family is simple and scalable, but previous versions suffered from boundary bugs, weak cell ordering, and bad inter-cell bridges.",
        "strict_constraints": [
            "The grid/sector structure must define the macro-order of the tour.",
            "Use snake-like, radial, sweep, or endpoint-bridging order between cells/sectors.",
            "Do not just run nearest-neighbor over all cities after computing the cells.",
            "Fix boundary cases where max-coordinate cities fall outside the last grid cell.",
            "Local paths and inter-region bridges must both be handled explicitly.",
        ],
    },
    {
        "id": "sparse_geometric_graph",
        "enabled": False,
        "name": "Sparse geometric graph / Gabriel-Delaunay approximation",
        "objective": "Build a bounded-degree geometric graph from coordinates, then construct fragments or paths from that graph and patch them into a tour.",
        "description": "Previous Delaunay/Gabriel attempts were conceptually interesting but often fake, disconnected, or O(n^3). This block tests whether a scalable approximate sparse graph can guide construction.",
        "strict_constraints": [
            "Build a bounded-degree geometric graph using only scalable local approximations.",
            "Do not use scipy, networkx, sklearn, or external triangulation libraries.",
            "Do not scan all triples and do not build an O(n^3) Gabriel graph.",
            "The graph must guide path/fragment construction, not merely be computed and ignored.",
            "Include a fallback for disconnected sparse graph components that preserves the same graph/fragment idea.",
        ],
    },
    {
        "id": "clustering_decomposition",
        "enabled": False,
        "name": "Cluster-based decomposition",
        "objective": "Cluster cities into spatial groups, build local tours or paths in each cluster, then connect clusters through endpoint-aware bridges.",
        "description": "Historical runs often produced cluster names but then used nearest-neighbor inside each cluster and concatenated clusters poorly. This block focuses on real decomposition and bridge quality.",
        "strict_constraints": [
            "The cluster decomposition must be used to define subproblems and bridges.",
            "Do not simply concatenate cluster tours in arbitrary cluster order.",
            "The algorithm must choose cluster order or cluster bridges using endpoint costs or geometric adjacency.",
            "Avoid expensive iterative k-means loops or dense all-pairs assignment when possible.",
            "Do not let the local solver become the whole method; the inter-cluster bridge logic matters.",
        ],
    },
    {
        "id": "spectral_clustering_proxy",
        "enabled": False,
        "name": "Spectral clustering / graph-cut proxy",
        "objective": "Approximate a graph-cut or spectral-style partition without dense eigen-decomposition, then build and bridge region paths.",
        "description": "The LLM attempted spectral clustering before, but dense matrices and eigenvectors were not scalable. This block keeps the partitioning idea while forbidding dense spectral machinery.",
        "strict_constraints": [
            "Do not build a dense n by n distance or affinity matrix.",
            "Do not call numpy eigen-decomposition on an n by n Laplacian.",
            "Use a scalable proxy for spectral/graph-cut behavior, such as coordinate projections, recursive bisection, sparse adjacency, or region cuts.",
            "The partition must feed into local path construction and endpoint bridging.",
            "Do not rename a simple nearest-neighbor tour as spectral clustering.",
        ],
    },
    {
        "id": "polar_angle_sweep",
        "enabled": False,
        "name": "Polar-angle / sweep construction",
        "objective": "Use angular or radial ordering around one or more centers to construct a global sweep tour, with explicit handling of radius jumps and endpoint bridges.",
        "description": "Pure polar sorting was weak in previous runs, but sweep-style construction is scalable and distinct. This block tests whether the LLM can repair the radius-jump problem.",
        "strict_constraints": [
            "The main ordering must be based on angular/radial sweep logic, not closest-unvisited choice.",
            "Handle the common failure where nearby angles but very different radii create long edges.",
            "Consider multi-center sweep, radial bins, alternating sweep direction, or endpoint-aware sector bridges.",
            "Do not sort by angle and then ignore the order by selecting the nearest city anyway.",
            "Bounded cleanup is allowed only after the sweep tour is built.",
        ],
    },
    {
        "id": "region_growth_endpoint_bridging",
        "enabled": False,
        "name": "Region growth / endpoint-bridging construction",
        "objective": "Grow several local path fragments or regions and repeatedly connect endpoints using bridge scarcity, region adjacency, and closure-risk rules.",
        "description": "This is the general region/path-fragment idea that appeared in the avoidance runs. The focus is on endpoint management rather than inserting single cities or walking to nearest neighbors.",
        "strict_constraints": [
            "Maintain multiple open path endpoints or region endpoints during construction.",
            "The main decision must be which endpoints/fragments to connect next, not which single city is closest to the current city.",
            "Avoid subtours and degree violations while building fragments.",
            "Use bridge scarcity, endpoint distance, region adjacency, or future closure risk to choose connections.",
            "Do not reduce this to a single current-city greedy walk.",
        ],
    },
]

# -------------------------
# Evaluation/runtime behavior
# -------------------------
GLOBAL_SEED = 12345
EVAL_SPLIT = "train"               # "train", "val", "test", or "all"; train = original feedback split
CANDIDATE_TIMEOUT_S = 60

# -------------------------
# POPMUSIC / sparse-candidate controls
# -------------------------
# This is the TSP equivalent of the clustering repo's big mode toggles:
# one variable switch should make it obvious whether candidate/prior mode is active.
USE_POPMUSIC_CANDIDATES = True
USE_POPMUSIC_EDGE_PRIOR = True
POPMUSIC_PRIOR_MODE = "frequency"  # "none", "frequency", "binary_topk", "shuffled"
MAX_CANDIDATES = 20
LKH_BINARY_PATH = "/content/tools/lkh/LKH"

# Historical edge-prior methodology:
# 30 short LKH/POPMUSIC runs, TIME_LIMIT=1.0s, MOVE_TYPE=5, PATCHING_A=2, PATCHING_C=3.
EDGE_PRIOR_RUNS = 30
EDGE_PRIOR_TIME_LIMIT_S = 1.0
EDGE_PRIOR_TOPK = 5
EDGE_PRIOR_FORCE_REBUILD = False

# -------------------------
# Drive paths
# -------------------------
INSTANCE_ROOT = "/content/drive/MyDrive/TM/TSP_instances"
CANDIDATE_CACHE_DIR = "/content/drive/MyDrive/TM/LKH_candidate_cache"
EDGE_PRIOR_CACHE_DIR = "/content/drive/MyDrive/TM/LKH_edge_prior_cache"
ARTIFACT_ROOT = "/content/drive/MyDrive/TM/llm-tsp-runs"

# -------------------------
# Original LLM-loop TSPLIB feedback suite
# -------------------------
# These are the same instances used for the thesis LLM feedback loop.
# Large final-evaluation/scaling instances are intentionally not included here.
TRAIN_INSTANCES = ["dsj1000", "pr1002", "d1291"]
VAL_INSTANCES = ["fl1400", "pcb1173"]
TEST_INSTANCES = ["rl1304", "u1817"]
OPTIMA = {
    "dsj1000": 18659688,
    "pr1002": 259045,
    "d1291": 50801,
    "fl1400": 20127,
    "pcb1173": 56892,
    "rl1304": 252948,
    "u1817": 57201,
}

# -------------------------
# Notebook behavior
# -------------------------
AUTO_DOWNLOAD_ARTIFACT_ZIP = True
COPY_ZIP_TO_DRIVE = True


## 2. Mount Google Drive

The TSPLIB files are read from `INSTANCE_ROOT`, POPMUSIC candidate files from `CANDIDATE_CACHE_DIR`, LKH tour-frequency priors from `EDGE_PRIOR_CACHE_DIR`, and run artifacts are saved under `ARTIFACT_ROOT`.


In [20]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Clone or refresh the GitHub backend repo

The repository is public, so the notebook clones it directly from GitHub.


In [21]:
repo_dir = f"/content/{REPO}"
repo_url = f"https://github.com/{ORG}/{REPO}.git"

# Move out of the repo before deleting/recloning it.
%cd /content
!rm -rf "{repo_dir}"
!git clone --branch "{BRANCH}" "{repo_url}" "{repo_dir}"

%cd "{repo_dir}"
!ls


/content
Cloning into '/content/llm-tsp-heuristics'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (284/284), done.
remote: Total 434 (delta 207), reused 354 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (434/434), 208.75 KiB | 6.73 MiB/s, done.
Resolving deltas: 100% (207/207), done.
/content/llm-tsp-heuristics
configs  docs	      notebooks       README.md		scripts      src
data	 experiments  pyproject.toml  requirements.txt	server_eval  tests


## 4. Install backend package

This installs the local package in editable mode without upgrading Colab runtime packages.

Important: we intentionally use `--no-deps` here to avoid Colab restart prompts caused by upgrading packages such as `ipython`, `ipykernel`, `tornado`, or `prompt-toolkit`. The notebook only installs truly missing minimal dependencies.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path.cwd()
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Install the repo itself without dependencies. This avoids Colab runtime restart prompts.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    check=True,
)

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "PyYAML",
}
missing_packages = [pkg for module, pkg in required_modules.items() if importlib.util.find_spec(module) is None]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
else:
    print("All minimal runtime dependencies already available.")

import llm_tsp
print("llm_tsp import OK from:", llm_tsp.__file__)


## 5. Build runtime config from the control panel

This cell intentionally stays short. The conversion from the top control-panel variables to a temporary YAML file lives in `src/llm_tsp/notebook_runtime.py`.


In [ ]:
# No large-instance download/build step in the LLM loop.
# The launcher uses the original feedback instances only.
# If a final-evaluation cache is needed later, build it separately on the school server.
print("Skipping large-instance download/cache-build cell by design.")

import sys
import importlib
from pathlib import Path

REPO_DIR = Path(f"/content/{REPO}")
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import llm_tsp.notebook_runtime as notebook_runtime
importlib.reload(notebook_runtime)

RUNTIME_CONFIG_PATH, EFFECTIVE_CONFIG = notebook_runtime.build_runtime_config_from_notebook_globals(globals())

print("Runtime config:", RUNTIME_CONFIG_PATH)
print("Module file used:", notebook_runtime.__file__)
notebook_runtime.print_effective_config(EFFECTIVE_CONFIG)


## Check selected TSPLIB files

This cell verifies that the selected original LLM-loop instances already exist under `INSTANCE_ROOT`. It does not download missing files.


In [ ]:
from pathlib import Path

instance_root = Path(INSTANCE_ROOT)
selected_names = notebook_runtime.selected_instance_names(EFFECTIVE_CONFIG)

print("Selected instances:", selected_names)
print("INSTANCE_ROOT:", instance_root)
print("DOWNLOAD_MISSING_TSPLIB:", DOWNLOAD_MISSING_TSPLIB)

missing = []
for name in selected_names:
    found = next((p for p in notebook_runtime.tsplib_file_candidates(name, instance_root) if p.exists()), None)
    if found:
        print("OK     ", name, found)
    else:
        print("MISSING", name)
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Missing required TSPLIB files under INSTANCE_ROOT: " + ", ".join(missing) +
        ". This LLM launcher does not download/build files; copy them locally or build them on the server."
    )

print("All selected TSPLIB files are present.")


## 6. Check required Drive files

This repo intentionally ignores the old 100-city instances. The active suite is only the 1k+ TSPLIB set:

- train: `dsj1000`, `pr1002`, `d1291`
- val: `fl1400`, `pcb1173`
- test: `rl1304`, `u1817`


In [ ]:
from pathlib import Path

selected_names = notebook_runtime.selected_instance_names(EFFECTIVE_CONFIG)
instance_root = Path(INSTANCE_ROOT)
candidate_root = Path(CANDIDATE_CACHE_DIR)
edge_prior_root = Path(EDGE_PRIOR_CACHE_DIR)

print("Selected instances:", selected_names)
print("INSTANCE_ROOT:", instance_root)
print("CANDIDATE_CACHE_DIR:", candidate_root)
print("EDGE_PRIOR_CACHE_DIR:", edge_prior_root)

missing_instances = []
missing_candidates = []
missing_priors = []

for name in selected_names:
    tsp_candidates = notebook_runtime.tsplib_file_candidates(name, instance_root)
    found_tsp = next((path for path in tsp_candidates if path.exists()), None)
    print(("OK      " if found_tsp else "MISSING "), name, "TSPLIB", found_tsp or tsp_candidates[0])
    if not found_tsp:
        missing_instances.append(name)

    if USE_POPMUSIC_CANDIDATES:
        cand_candidates = notebook_runtime.candidate_file_candidates(name, candidate_root)
        found_cand = next((path for path in cand_candidates if path.exists()), None)
        print(("OK      " if found_cand else "MISSING "), name, "candidate", found_cand or cand_candidates[0])
        if not found_cand:
            missing_candidates.append(name)

    if USE_POPMUSIC_EDGE_PRIOR:
        prior_candidates = notebook_runtime.edge_prior_file_candidates(name, edge_prior_root)
        found_prior = next((path for path in prior_candidates if path.exists()), None)
        print(("OK      " if found_prior else "MISSING "), name, "edge-prior", found_prior or prior_candidates[0])
        if not found_prior:
            missing_priors.append(name)

if missing_instances:
    raise FileNotFoundError("Missing required TSPLIB files: " + ", ".join(missing_instances))

if USE_POPMUSIC_CANDIDATES and missing_candidates:
    print("Candidate files are missing. The backend will generate them with LKH/POPMUSIC during problem loading.")
if USE_POPMUSIC_EDGE_PRIOR and missing_priors:
    print("Edge-prior files are missing. The backend will generate them with 30 short LKH/POPMUSIC runs during problem loading.")
if (not missing_candidates) and (not missing_priors):
    print("All selected cache files found.")


## Build POPMUSIC candidate lists and edge-prior caches

This cell is the important one for the final server evaluation. It builds missing `.cand` and `.npz` files in Drive using the same historical parameters used by the TSP pipeline. Existing cache files are skipped unless `EDGE_PRIOR_FORCE_REBUILD=True`.


In [ ]:
from pathlib import Path

# In the LLM loop, never generate POPMUSIC candidate files or edge-prior caches.
# We only verify that the files needed by the selected original feedback instances exist.
selected_names = notebook_runtime.selected_instance_names(EFFECTIVE_CONFIG)
candidate_root = Path(CANDIDATE_CACHE_DIR)
edge_prior_root = Path(EDGE_PRIOR_CACHE_DIR)

needs_candidates = bool(USE_POPMUSIC_CANDIDATES or USE_POPMUSIC_EDGE_PRIOR)
needs_prior = bool(USE_POPMUSIC_EDGE_PRIOR)

print("Cache generation is disabled in this launcher.")
print("needs_candidates:", needs_candidates)
print("needs_edge_prior:", needs_prior)

missing_candidates = []
missing_priors = []

for name in selected_names:
    if needs_candidates:
        cand = next((p for p in notebook_runtime.candidate_file_candidates(name, candidate_root) if p.exists() and p.stat().st_size > 0), None)
        if cand:
            print("candidate OK", name, cand)
        else:
            print("candidate MISSING", name)
            missing_candidates.append(name)
    if needs_prior:
        prior = next((p for p in notebook_runtime.edge_prior_file_candidates(name, edge_prior_root) if p.exists() and p.stat().st_size > 0), None)
        if prior:
            print("prior OK    ", name, prior)
        else:
            print("prior MISSING", name)
            missing_priors.append(name)

if missing_candidates or missing_priors:
    msg = []
    if missing_candidates:
        msg.append("missing candidate files for: " + ", ".join(missing_candidates))
    if missing_priors:
        msg.append("missing edge-prior files for: " + ", ".join(missing_priors))
    raise FileNotFoundError(
        "; ".join(msg) +
        ". This LLM launcher does not generate cache files. Build/copy them separately before launching."
    )

print("All required candidate/prior files are present for the selected feedback instances.")


## 7. Load Groq API keys

This cell first tries Colab Secrets via `google.colab.userdata`, then falls back to manual input.

Expected secret names are `GROQ_API_KEY_1`, `GROQ_API_KEY_2`, etc.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

key_names = [f"GROQ_API_KEY_{i}" for i in range(1, int(GROQ_MAX_KEYS) + 1)]
loaded = []

for key_name in key_names:
    if os.environ.get(key_name):
        loaded.append(key_name)
        continue

    val = None
    if userdata is not None:
        try:
            val = userdata.get(key_name)
        except Exception:
            val = None

    if val:
        os.environ[key_name] = val
        loaded.append(key_name)

print("Loaded Groq keys from environment/Colab Secrets:", loaded)

if not os.environ.get("GROQ_API_KEY_1"):
    os.environ["GROQ_API_KEY_1"] = getpass("Paste GROQ_API_KEY_1 manually: ")

print("GROQ_API_KEY_1 found:", bool(os.environ.get("GROQ_API_KEY_1")))


## 8. Run the pipeline with live logs

This streams backend logs while the pipeline runs and prints clustering-style per-call feedback:

- `name` and backend-inferred heuristic `family`
- per-instance validity/cost/gap/runtime feedback
- invalid/partial error excerpts
- final best-candidate summary

Artifacts are written with clustering-style names: `llm_attempts.csv`, `llm_search_instance_rows.csv`, `llm_best_attempts_top20.csv`, `llm_family_summary.csv`, `best_candidate_code.py`, and `best_candidate_summary.json`.


In [ ]:
import subprocess
import sys

if CACHE_ONLY_BUILD_TSP_CACHES:
    print("CACHE_ONLY_BUILD_TSP_CACHES=True")
    print("Skipping LLM pipeline launch. Candidate/prior caches have been built or verified above.")
    PIPELINE_LOG_LINES = []
    LATEST_RUN_DIR = None
else:
    print("Launching TSP pipeline")
    print("Using runtime config:", RUNTIME_CONFIG_PATH)
    print("Live pipeline logs will appear below.")
    print("-" * 100)

    cmd = [sys.executable, "-u", "scripts/run_unified_tsp_pipeline.py", "--config", RUNTIME_CONFIG_PATH]

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    PIPELINE_LOG_LINES = []
    for line in process.stdout:
        PIPELINE_LOG_LINES.append(line)
        print(line, end="")

    return_code = process.wait()
    print("-" * 100)
    print("Pipeline return code:", return_code)

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


## 9. Locate latest run folder and summarize artifacts

This finds the latest run folder matching `RUN_NAME` and lists the main generated artifacts.


In [ ]:
from pathlib import Path
import re

if CACHE_ONLY_BUILD_TSP_CACHES:
    print("CACHE_ONLY_BUILD_TSP_CACHES=True, no LLM run folder to locate.")
    print("Cache log:", Path(ARTIFACT_ROOT) / "tsp_cache_check_log.csv")
else:
    runs_dir = Path(ARTIFACT_ROOT)
    print("Runs directory:", runs_dir)

    artifact_from_log = None
    for line in PIPELINE_LOG_LINES:
        if line.startswith("ARTIFACT_DIR:"):
            artifact_from_log = Path(line.split(":", 1)[1].strip())
            break

    if artifact_from_log and artifact_from_log.exists():
        LATEST_RUN_DIR = artifact_from_log
    else:
        if not runs_dir.exists():
            raise FileNotFoundError(f"Runs directory does not exist: {runs_dir}")
        matching_dirs = [p for p in runs_dir.iterdir() if p.is_dir() and p.name.startswith(RUN_NAME)]
        all_dirs = [p for p in runs_dir.iterdir() if p.is_dir()]
        if matching_dirs:
            LATEST_RUN_DIR = max(matching_dirs, key=lambda p: p.stat().st_mtime)
        elif all_dirs:
            LATEST_RUN_DIR = max(all_dirs, key=lambda p: p.stat().st_mtime)
        else:
            raise FileNotFoundError(f"No run folders found in {runs_dir}")

    print("Latest run dir:", LATEST_RUN_DIR)

    interesting_suffixes = {".csv", ".jsonl", ".json", ".txt", ".py", ".yaml", ".yml"}
    for p in sorted(LATEST_RUN_DIR.rglob("*")):
        if p.is_file() and p.suffix.lower() in interesting_suffixes:
            print(p.relative_to(LATEST_RUN_DIR))


## 10. Quick result tables

This displays the most important summary CSVs if they exist.


In [ ]:
import pandas as pd
from IPython.display import display

if CACHE_ONLY_BUILD_TSP_CACHES:
    cache_log_path = Path(ARTIFACT_ROOT) / "tsp_cache_check_log.csv"
    print("CACHE_ONLY_BUILD_TSP_CACHES=True")
    print("Cache log exists:", cache_log_path.exists(), cache_log_path)
    if cache_log_path.exists():
        display(pd.read_csv(cache_log_path))
else:
    summary_files = [
        "selected_instances.csv",
        "search_instances.csv",
        "llm_attempts.csv",
        "llm_best_attempts_top20.csv",
        "llm_family_summary.csv",
        "llm_search_instance_rows.csv",
        "generated_attempts.csv",  # legacy alias kept for convenience
    ]

    for name in summary_files:
        path = LATEST_RUN_DIR / name
        if path.exists():
            print("\n===", name, "===")
            df = pd.read_csv(path)
            display(df.head(20))
        else:
            print("Missing:", name)

    best_summary = LATEST_RUN_DIR / "best_candidate_summary.json"
    best_code = LATEST_RUN_DIR / "best_candidate_code.py"
    print("\nBest summary exists:", best_summary.exists(), best_summary)
    print("Best code exists:", best_code.exists(), best_code)


## 11. Zip and auto-download latest artifact folder

If `AUTO_DOWNLOAD_ARTIFACT_ZIP = True`, this creates a zip of the latest run folder and triggers a browser download to your local PC.

If `COPY_ZIP_TO_DRIVE = True`, the zip is also copied back into `ARTIFACT_ROOT`.


In [ ]:
from pathlib import Path
import shutil
import zipfile

from google.colab import files

if CACHE_ONLY_BUILD_TSP_CACHES:
    print("CACHE_ONLY_BUILD_TSP_CACHES=True, no LLM run folder to zip.")
    print("No cache files were generated by this notebook. Existing cache dirs are:")
    print(" -", CANDIDATE_CACHE_DIR)
    print(" -", EDGE_PRIOR_CACHE_DIR)
    print("Cache log:", Path(ARTIFACT_ROOT) / "tsp_cache_check_log.csv")
elif AUTO_DOWNLOAD_ARTIFACT_ZIP:
    zip_base = Path("/content") / LATEST_RUN_DIR.name
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=LATEST_RUN_DIR.parent, base_dir=LATEST_RUN_DIR.name))

    print("Created zip:", zip_path)
    print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))

    if COPY_ZIP_TO_DRIVE:
        drive_zip = Path(ARTIFACT_ROOT) / zip_path.name
        shutil.copy2(zip_path, drive_zip)
        print("Copied zip to Drive:", drive_zip)

    print("\nIncluded files:")
    with zipfile.ZipFile(zip_path) as zf:
        for item in sorted(zf.namelist()):
            if not item.endswith("/"):
                # Print relative paths inside the run folder, like the clustering launcher does.
                rel = item.split("/", 1)[1] if "/" in item else item
                print(" -", rel)

    print("Starting browser download...")
    files.download(str(zip_path))
else:
    print("AUTO_DOWNLOAD_ARTIFACT_ZIP = False, skipping local download.")
    print("Latest run folder:", LATEST_RUN_DIR)
